**Radio-frequency interference mitigation in pulsar observations**

This notebook implements a multi-stage RFI-mitigation pipeline for a dedispersed pulsar filterbank observation. It combines frequency-domain and time-domain approaches to identify and mitigate interference while preserving the pulsar signal.

The pipeline includes channel-based RFI identification, DM = 0 discrimination, FFT-domain filtering, off-pulse time-domain cleaning, and a comparison of the resulting pulse profiles and signal-to-noise ratios.


In [ ]:
!pip install git+https://github.com/thepetabyteproject/your.git

**Install dependencies - Install environment**

Install the `your` pulsar-processing package from GitHub. The notebook also uses NumPy, SciPy, Matplotlib and Astropy for numerical processing, filtering, visualisation and statistical clipping.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.signal import detrend, savgol_filter
import your
from your.candidate import Candidate
from astropy.stats import sigma_clip
from your.utils.rfi import spectral_kurtosis

**Load observational data**

Update the directory path to match your own environment.

In [ ]:
#Replace with your own filepath (this is if your data is on google drive)
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/

In [ ]:
fil_file = ""                       # path to your .fil file
dm       =                          # dispersion measure (pc cm⁻³)
period   =                          # pulse period (samples)

# Pipeline tuning
sk_thresh   = 3.5     # SK flagging threshold (σ)
chunk       = 8192    # time samples per SK block (tune: 2048–8192 works well)
n_ave       = 2500    # block size for time-domain masking (2300-2500 gives the best SNR)
clip_sigma  = 6       # per-channel sigma-clip threshold

**Signal-processing and RFI-mitigation helper functions**

Define reusable functions for identifying outliers, filtering interference in the Fourier domain, estimating the pulse window, generating off-pulse masks and folding the time series.

In [ ]:
def Spurious_filter(data, window, sigma): #Identifies deviations from a Savitzky–Golay-smoothed signal.
  y = savgol_filter(data, window, 2)
  sub = data - y
  sigma = sigma * np.std(sub)
  return (sub > sigma) | (sub < -sigma)

def fft_rfi_clean(timeseries, chunk=10000, window=101, sigma=3): #Applies FFT-domain filtering to sections of a time series.
    timeseries = np.asarray(timeseries)
    cleaned = timeseries.copy()

    n = len(timeseries)
    for i in range(0, n, chunk):
        seg = timeseries[i:i+chunk]
        if seg.size < 32:
            continue

        z = np.fft.fft(seg)

        spec = np.log(np.abs(z[1:]) + 1e-12)  # log is much stabler
        w = min(window, spec.size if spec.size % 2 == 1 else spec.size - 1)
        if w < 5:
            cleaned[i:i+seg.size] = np.fft.ifft(z).real
            continue

        mask = Spurious_filter(spec, w, sigma)
        z2 = z.copy()
        z2[1:][mask] = 0

        cleaned[i:i+seg.size] = np.fft.ifft(z2).real

    return cleaned

def find_pulse_window(profile, width=40): #Estimates the pulse window from the folded profile.
    peak = np.argmax(profile)
    start = max(0, peak - width)
    end = min(len(profile), peak + width)
    return start, end

def find_on_pulse(profile, width=40): #Returns the phase range around the pulse peak.
    peak = np.argmax(profile)
    return max(0, peak - width), min(len(profile), peak + width)

def fold(timeseries, period): #Folds a time series using an integer period.
    P = int(period)
    timeseries = np.asarray(timeseries)

    prof = np.zeros(P, dtype=np.float64)
    nfold = int(len(timeseries) / period) - 2

    for i in range(nfold):
        idx = int(i * period)
        seg = timeseries[idx:idx+P]
        if seg.size != P:
            break
        prof += seg

    return prof

def make_offpulse_mask(timeseries, period, width=180): #Identifies off-pulse samples using the folded pulse profile.
    P = int(period)
    profile = fold(timeseries, period)
    peak = int(np.argmax(profile))

    on_start = (peak - width) % P
    on_end   = (peak + width) % P

    phase = np.arange(len(timeseries)) % P

    if on_start <= on_end:
        onpulse = (phase >= on_start) & (phase <= on_end)
    else:
        onpulse = (phase >= on_start) | (phase <= on_end)

    offpulse = ~onpulse
    return offpulse, (on_start, on_end)

def fold_float_step(timeseries, period): #Folds the time series while retaining the fractional period in the starting-index calculation.
    P = int(period)
    prof = np.zeros(P)
    nfold = int(len(timeseries) / period) - 2

    for i in range(nfold):
        start = int(i * period)          # float-step
        prof += timeseries[start:start+P]

    return prof

**Main RFI mitigation pipeline**

Load the filterbank observation, dedisperse the data at the pulsar DM, and construct the raw time series for comparison. The following stages identify and mitigate interference before producing a frequency-collapsed time series.

In [ ]:
your_object = your.Your(fil_file)
header = your_object.your_header
Ns = int(header.native_nspectra)
Nt = header.nspectra
tsamp = header.native_tsamp

cand = Candidate(
    fp=fil_file,
    dm=dm,
    tcand=int(Ns * tsamp / 2),
    min_samp=Ns
)

cand.get_chunk()
cand.dedisperse()
# --- PRISTINE raw path (must never be modified) ---
data_dd_raw = cand.dedispersed.copy()              # untouched DM-dedispersed data
raw_ts = np.mean(data_dd_raw, axis=1).astype(np.float32)  # EXACT "raw" collapse

# --- working copy for masking pipeline (can be modified freely) ---
data_dd = data_dd_raw.copy()

# ============================================================
# MEMORY-SAFE SPECTRAL KURTOSIS (CHUNKED)
# ============================================================

#data_dd = cand.dedispersed

Nt, Nf = data_dd.shape

edge = 3  # number of edge channels to protect

valid_chans = np.ones(Nf, dtype=bool)
valid_chans[:edge] = False
valid_chans[-edge:] = False

sum_p  = np.zeros(Nf)
sum_p2 = np.zeros(Nf)
count  = 0

for i in range(0, Nt, chunk):
    block = data_dd[i:i+chunk]

    # Skip tiny last block
    if block.shape[0] < 32:
        continue

    # Compute power statistics
    sum_p  += np.mean(block, axis=0)
    sum_p2 += np.mean(block**2, axis=0)
    count  += 1

# Compute SK
mean_p  = sum_p / count
mean_p2 = sum_p2 / count

sk_freq = mean_p2 / (mean_p**2 + 1e-12) - 1

# Threshold
freq_mask = (np.abs(sk_freq) > sk_thresh) & valid_chans

print(f"SK flagged {freq_mask.sum()} / {Nf} channels")

# Replace flagged channels
#data_dd[:, freq_mask] = np.median(data_dd[:, ~freq_mask])

#interpolate flagged channels instead of flattening ----
good = ~freq_mask
bad  = freq_mask

for t in range(0, Nt, 4096):  # chunk in time to stay memory-safe
    sl = slice(t, min(t+4096, Nt))

    band = data_dd[sl]

    for i in np.where(bad)[0]:
        left = i - 1
        right = i + 1

        while left >= 0 and bad[left]:
            left -= 1
        while right < Nf and bad[right]:
            right += 1

        if left >= 0 and right < Nf:
            band[:, i] = 0.5 * (band[:, left] + band[:, right])

# ============================================================
# END MEMORY-SAFE SK
# ============================================================

#This is just to check the bandpass shape after spectral kurtosis

#spec = np.mean(data_dd, axis=0)
#plt.figure(figsize=(6,3))
#plt.plot(spec)
#plt.xlabel("Frequency channel")
#plt.ylabel("Mean power")
#plt.title("Bandpass after SK")
#plt.show()

# ============================================================
# DM=0 RFI DISCRIMINATION
# ============================================================

cand_dm0 = Candidate(
    fp=fil_file,
    dm=0.0,
    tcand=int(Ns * tsamp / 2),
    min_samp=Ns
)
cand_dm0.get_chunk()
cand_dm0.dedisperse()

# Power time series
power_dm  = raw_ts
power_dm0 = np.mean(cand_dm0.dedispersed, axis=1)

# DM=0 ratio
dm0_ratio = power_dm0 / (power_dm + 1e-6)

# Robust threshold (PRESTO-like)
med = np.median(dm0_ratio)
mad = np.median(np.abs(dm0_ratio - med))
threshold = med + 8 * mad

dm0_rfi_mask = dm0_ratio > threshold

print(f"DM=0 masking {dm0_rfi_mask.sum()} / {len(dm0_rfi_mask)} samples")

# Apply mask to dedispersed data

# Collapse frequency
#time_series = np.mean(data_dd, axis=1)

ts_mask_in = np.zeros(Nt, dtype=np.float32)

# Per-channel clipping controls (tune later)
clip_maxiters = 3
max_chans_clip = 128   # clip at most this many channels per block (speed control)

for i in range(Nt // n_ave):
    sl = slice(i*n_ave, (i+1)*n_ave)

    block2d = data_dd[sl]  # view: shape (n_ave, Nf)

    # Bandpass estimate + dynamic bad-channel mask
    pws = block2d.mean(axis=0)

    pws0 = pws - np.median(pws)
    mad = np.median(np.abs(pws0)) + 1e-12
    pwsn = pws0 / mad

    f_mask = Spurious_filter(pwsn, 7, 3)
    good = ~f_mask
    good_idx = np.where(good)[0]

    if good_idx.size < 16:
        ts_mask_in[sl] = block2d.mean(axis=1).astype(np.float32)
        continue

    # Compute per-channel scatter
    scat = np.std(block2d[:, good_idx], axis=0)
    if good_idx.size > max_chans_clip:
        worst = np.argsort(scat)[-max_chans_clip:]
        clip_idx = good_idx[worst]
        no_clip_idx = np.setdiff1d(good_idx, clip_idx, assume_unique=False)
    else:
        clip_idx = good_idx
        no_clip_idx = np.array([], dtype=int)

    # Start with sum of channels we won't clip
    ts_sum = np.sum(block2d[:, no_clip_idx], axis=1, dtype=np.float32) if no_clip_idx.size else np.zeros(block2d.shape[0], dtype=np.float32)

    # Add clipped versions of the selected channels
    for j in clip_idx:
        x = block2d[:, j]  # 1D view length n_ave
        filt = sigma_clip(x, sigma=clip_sigma, maxiters=clip_maxiters)
        if np.any(filt.mask):
            x2 = x.copy()  # small copy: only length n_ave
            x2[filt.mask] = np.median(x[~filt.mask])
            ts_sum += x2.astype(np.float32)
        else:
            ts_sum += x.astype(np.float32)

    # Final collapse for this block
    ts_mask_in[sl] = ts_sum / good_idx.size

# DM=0 masking ONLY in time domain
ts_median = np.median(ts_mask_in[~dm0_rfi_mask])
ts_mask_in[dm0_rfi_mask] = ts_median

offpulse_mask, on_window_est = make_offpulse_mask(ts_mask_in, period, width=0)
print(f"On-pulse window (estimate): {on_window_est}")

del cand_dm0
import gc
gc.collect()

del cand
gc.collect()

# ============================================================
# END DM=0 RFI DISCRIMINATION
# ============================================================


# =========================
# STEP 1: FFT-domain RFI removal
# =========================

# FFT clean
time_fftclean = ts_mask_in.copy()

# FFT clean ONLY off-pulse samples
#time_fftclean[offpulse_mask] = fft_rfi_clean(
 #   ts_mask_in[offpulse_mask],
  #  chunk=4096)

off_med = np.median(ts_mask_in[offpulse_mask])
ts_for_fft = ts_mask_in.copy()
ts_for_fft[~offpulse_mask] = off_med  # neutralize on-pulse

ts_for_fft_clean = fft_rfi_clean(ts_for_fft, chunk=4096)

time_fftclean[offpulse_mask] = ts_for_fft_clean[offpulse_mask]
time_fftclean[~offpulse_mask] = ts_mask_in[~offpulse_mask]

# Apply FFT-cleaning back to 2D data
#weights = time_fftclean / np.mean(time_fftclean)
#data_fftclean = data_dd * weights[:, None]


**Off-pulse time-domain RFI mitigation**

Apply additional statistical filtering to the frequency-collapsed time series. The filtering is restricted to off-pulse samples to reduce the risk of modifying the pulsar signal.

In [ ]:
# =========================
# STEP 2: Time-domain statistical RFI masking (OFF-PULSE ONLY)
# =========================

clean_time_series = time_fftclean.copy()

for i in range(len(clean_time_series) // n_ave):
    sl = slice(i*n_ave, (i+1)*n_ave)

    block = clean_time_series[sl]
    off_block = offpulse_mask[sl]

    n_off = np.sum(off_block)
    if n_off < 300:
      continue

    off_med = np.median(block[off_block])
    block_for_filter = block.copy()
    block_for_filter[~off_block] = off_med

    # Run Spurious_filter on the CONTIGUOUS block
    mask_full = Spurious_filter(block_for_filter, window=201, sigma=3)

    # Only apply flagged replacements to OFF-pulse samples
    apply_idx = np.where(off_block & mask_full)[0]
    if apply_idx.size:
      block[apply_idx] = off_med

    clean_time_series[sl] = block

# =========================
# STEP 3: Final sigma clipping (OFF-PULSE ONLY)
# =========================

off_vals = clean_time_series[offpulse_mask]

mu = np.median(off_vals)
sigma = np.std(off_vals)

# only remove spikes clearly above noise
mask_pos = off_vals > (mu + 3*sigma)

print("Positive spikes clipped:", np.sum(mask_pos))

final_timeseries = clean_time_series.copy()
off_vals_new = off_vals.copy()

# replace spikes with random Gaussian noise to preserve statistics
rng = np.random.default_rng()
off_vals_new[mask_pos] = rng.normal(mu, sigma, size=np.sum(mask_pos))

final_timeseries[offpulse_mask] = off_vals_new

# =========================
# STEP 4: Folding (raw vs masked)
# =========================

pulse_raw = detrend(fold_float_step(raw_ts, period))

pulse_masked = detrend(fold_float_step(final_timeseries, period))

# Align pulses
#peak = np.argmax(pulse_raw)
#offset = int(period/2) - peak

#pulse_raw = np.roll(pulse_raw, offset)
#pulse_masked = np.roll(pulse_masked, offset)

def compute_snr_sample(profile, off_bins):
    """
    Sample-script SNR:
    SNR = max(profile) / std(profile[off_bins])
    """
    profile = np.asarray(profile).squeeze()
    noise = np.std(profile[off_bins])
    if noise == 0:
        return np.nan
    return np.max(profile) / noise

# =========================
# STEP 5: SNR calculations
# =========================

def rebin_by_2(x):
    n = (len(x) // 2) * 2      # make it even
    return x[:n].reshape(n//2, 2).mean(1)

prof_raw    = rebin_by_2(pulse_raw)
prof_masked = rebin_by_2(pulse_masked)

# Align AFTER rebinning
peak = np.argmax(prof_raw)   # anchor alignment to raw
offset = (len(prof_masked) // 2) - peak

prof_raw    = np.roll(prof_raw, offset)
prof_masked = np.roll(prof_masked, offset)

# Off-pulse bins
nb = len(prof_raw)

off1 = np.arange(0, int(0.42*nb))
off2 = np.arange(int(0.56*nb), int(0.98*nb))
off_all = np.r_[off1, off2]

snr_raw_b1   = compute_snr_sample(prof_raw, off1)
snr_raw_b2   = compute_snr_sample(prof_raw, off2)
snr_raw_all  = compute_snr_sample(prof_raw, off_all)

snr_mask_b1  = compute_snr_sample(prof_masked, off1)
snr_mask_b2  = compute_snr_sample(prof_masked, off2)
snr_mask_all = compute_snr_sample(prof_masked, off_all)



# =========================
# Plot results
# =========================

plt.figure(figsize=(10,4))
plt.plot(prof_raw, label=f"Raw | SNR all={snr_raw_all:.1f}, 0–150={snr_raw_b1:.1f}, 200–350={snr_raw_b2:.1f}", lw=2)
plt.plot(prof_masked, label=f"Masked | SNR all={snr_mask_all:.1f}, 0–150={snr_mask_b1:.1f}, 200–350={snr_mask_b2:.1f}", lw=2)
plt.xlabel("Pulse phase")
plt.ylabel("Intensity")
plt.legend()
plt.tight_layout()
plt.show()

**Comparison of off-pulse noise statistics**

Compare the standard deviation of the off-pulse regions in the raw and mitigated pulse profiles. Also calculate the standard deviation after detrending to assess the change in baseline fluctuations.

In [ ]:
# --- Bin-averaged profiles (same as plotting + sample SNR) ---
prof_raw    = pulse_raw.reshape(357, 2).mean(1)
prof_masked = pulse_masked.reshape(357, 2).mean(1)

# --- Off-pulse bins (same as sample SNR) ---
off1 = np.arange(0, 150)
off2 = np.arange(200, 350)
off_all = np.r_[off1, off2]

off_raw = prof_raw[off_all]
off_m   = prof_masked[off_all]

print("Off-pulse std (raw):    ", np.std(off_raw))
print("Off-pulse std (masked): ", np.std(off_m))

print("Off-pulse detrended std (raw):    ", np.std(detrend(off_raw)))
print("Off-pulse detrended std (masked): ", np.std(detrend(off_m)))